In [1]:
import importlib
import sys

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [2]:
# Load decision-labeled data
file_path_train = '../../../../../../data/BPIC20_DD/tensor_data/decision_labeled/bpic20_dd_all_5_train.pkl'
bpic20_dd_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/BPIC20_DD/tensor_data/decision_labeled/bpic20_dd_all_5_val.pkl'
bpic20_dd_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# BPIC20_DD dataset dynamic categories, features:
bpic20_dd_all_categories = bpic20_dd_train_dataset.all_categories

bpic20_dd_all_categories_cat = bpic20_dd_all_categories[0]
# print(bpic20_dd_all_categories_cat)
bpic20_dd_all_categories_num = bpic20_dd_all_categories[1]
# print(bpic20_dd_all_categories_num)
for i, cat in enumerate(bpic20_dd_all_categories_cat):
     print(f"BPIC20_DD dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(bpic20_dd_all_categories_num):
     print(f"BPIC20_DD dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of numerical: {num[1]}")
print('\n')
     
# BPIC20_DD dataset static categories, features:
bpic20_dd_all_stat_categories = bpic20_dd_train_dataset.all_static_categories

bpic20_dd_all_stat_categories_cat = bpic20_dd_all_stat_categories[0]
# print(bpic20_dd_all_stat_categories_cat)
bpic20_dd_all_stat_categories_num = bpic20_dd_all_stat_categories[1]
# print(bpic20_dd_all_stat_categories_num)
for i, cat in enumerate(bpic20_dd_all_stat_categories_cat):
     print(f"BPIC20_DD static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(bpic20_dd_all_stat_categories_num):
     print(f"BPIC20_DD static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of numerical: {num[1]}")  

BPIC20_DD dynamic categorical feature: concept:name, Index position in categorical data list: 0
BPIC20_DD amount of category labels: 10
BPIC20_DD dynamic categorical feature: org:resource, Index position in categorical data list: 1
BPIC20_DD amount of category labels: 4
BPIC20_DD dynamic categorical feature: org:role, Index position in categorical data list: 2
BPIC20_DD amount of category labels: 9


BPIC20_DD dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: day_in_week, Index position in categorical data list: 2
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: seconds_in_day, Index position in categorical data list: 3
BPIC20_DD amount of numerical: 1


BPIC20_DD static categorical feature: case:BudgetNumber, Index position in

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)

# Encoder features (fixed): use only requested features
enc_feat_cat = ['concept:name', 'org:resource']
enc_feat_num = ['case_elapsed_time']
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Decoder features (unused by C-LSTM training cell, kept for consistency)
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

# Guard tensors are pre-computed and stored in the dataset .pkl files
# (prepared during data loading via prepare_guard_tensors).
print(f"Guard tensors: targets {bpic20_dd_train_dataset._guard_targets.shape}, "
      f"mask {bpic20_dd_train_dataset._guard_mask.shape}, "
      # is that still present?
      f"confidence {bpic20_dd_train_dataset._guard_confidence.shape}")

Input features encoder:  [['concept:name', 'org:resource'], ['case_elapsed_time']]
Features decoder:  [['concept:name'], []]
Guard tensors: targets torch.Size([36663, 20, 10]), mask torch.Size([36663, 20]), confidence torch.Size([36663, 20])


In [5]:
import suffix_pred.models.C_LSTM
importlib.reload(suffix_pred.models.C_LSTM)
from suffix_pred.models.C_LSTM import FullShared_Join_LSTM

# Size hidden layer
hidden_size = 50

# Number of LSTM cells
num_layers = 1

# STANDARD: One numerical output to predict
input_size = 1

# C-LSTM uses dynamic prefix features only
model_feat = enc_feat

# Output size: activity classes only
activity_feature_name = 'concept:name'
activity_feature_index = next(i for i, cat in enumerate(bpic20_dd_all_categories_cat) if cat[0] == activity_feature_name)
output_size_act = bpic20_dd_all_categories_cat[activity_feature_index][1]

# C-LSTM model initialization
model = FullShared_Join_LSTM(data_set_categories=bpic20_dd_all_categories,
                             hidden_size=hidden_size,
                             num_layers=num_layers,
                             model_feat=model_feat,
                             input_size=input_size,
                             output_size_act=output_size_act)

Data set categories:  ([('concept:name', 10, {'Declaration APPROVED': 1, 'Declaration FINAL_APPROVED': 2, 'Declaration FOR_APPROVAL': 3, 'Declaration REJECTED': 4, 'Declaration SAVED': 5, 'Declaration SUBMITTED': 6, 'EOS': 7, 'Payment Handled': 8, 'Request Payment': 9}), ('org:resource', 4, {'EOS': 1, 'STAFF MEMBER': 2, 'SYSTEM': 3}), ('org:role', 9, {'ADMINISTRATION': 1, 'BUDGET OWNER': 2, 'EMPLOYEE': 3, 'EOS': 4, 'MISSING': 5, 'PRE_APPROVER': 6, 'SUPERVISOR': 7, 'UNDEFINED': 8})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {})])
Model input features:  [['concept:name', 'org:resource'], ['case_elapsed_time']]


Embeddings:  ModuleList(
  (0): Embedding(10, 6)
  (1): Embedding(4, 3)
)
Total embedding feature size:  9
Input feature size:  10
Cells hidden size:  50
Number of LSTM layer:  1


/home/PSPLab/.local/share/virtualenvs/decision_aware_augmentation_for_PPM-0DzgvVpC/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [7]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import CTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 0.001
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)


# Keep consistent across all models
# Epochs / Batch size
num_epochs = 100
batch_size = 128

# Shuffle data
shuffle = True

optimize_values = {"optimizer": optimizer,
                   "scheduler": scheduler,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle,
                   # Optional the higher the smaller get S:
                   "guard_support_threshold": 0.0}

# Activity feature index and EOS id for activity-only next-event prediction
activity_feature_name = 'concept:name'
concept_name_id = next(i for i, cat in enumerate(bpic20_dd_all_categories_cat) if cat[0] == activity_feature_name)
activity_label_to_id = bpic20_dd_all_categories_cat[concept_name_id][2]
eos_id = activity_label_to_id.get('EOS')
if eos_id is None:
    raise ValueError("Could not find EOS id in activity label mapping.")

# Decision-aware guard regularization weight (λ_g)
lambda_g = 1.0
print(f"Decision-aware training: λ_g = {lambda_g}")

model_output_path = "../../../../../../models/BPIC20_DD/decision/BPIC20_DD_C_LSTM_v1_DA.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = CTraining(device=device,
                    model=model,
                    data_train=bpic20_dd_train_dataset,
                    data_val=bpic20_dd_val_dataset,
                    optimize_values=optimize_values,
                    concept_name_id=concept_name_id,
                    eos_id=eos_id,
                    loss_obj=loss_obj,
                    lambda_g=lambda_g,
                    save_model_n_th_epoch=1,
                    saving_path=model_output_path)

# Train the model (decision-aware: L_base + λ_g * L_guard)
trainer.train()

Device: cuda
Decision-aware training: λ_g = 1.0
Device:  cuda
Optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7efe2d88b860>
Epochs:  100
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.8590
Validation: Avg Validation Loss: 0.2117
Validation Loss for Scheduler: 0.2117
saving model
Epoch [2/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5419
Validation: Avg Validation Loss: 0.1702
Validation Loss for Scheduler: 0.1702
saving model
Epoch [3/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5371
Validation: Avg Validation Loss: 0.1580
Validation Loss for Scheduler: 0.1580
saving model
Epoch [4/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5826
Validation: Avg Validation Loss: 0.1647
Validation Loss for Scheduler: 0.1647
saving model
Epoch [5/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.4377
Validation: Avg Validation Loss: 0.1497
Validation Loss for Scheduler: 0.1497
saving model
Epoch [6/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5765
Validation: Avg Validation Loss: 0.1613
Validat

([0.8590481284395743,
  0.5418548660047794,
  0.5371051193685482,
  0.5825893294312813,
  0.43767371946355194,
  0.5765296993614905,
  0.5268712274159081,
  0.5500655523420211,
  0.4943011097272514,
  0.5161699967309573,
  0.47923147699143415,
  0.5429869465503958,
  0.5367136868268563,
  0.5463590367278571,
  0.470518437243401,
  0.49531303549944733,
  0.577347934830168,
  0.47937123686438654,
  0.5516493845640159,
  0.47078496944613574,
  0.4894746678787241,
  0.440624432320275,
  0.6133963383396743,
  0.5488245223846793,
  0.5242796132295597,
  0.43426164261711186,
  0.45744056222175056,
  0.43837793760702587,
  0.542765386532408,
  0.45929023709135186,
  0.4589786290447471,
  0.35993915325102077,
  0.5673700471502771,
  0.4624016105427767,
  0.4297056028210535,
  0.5174490945861522,
  0.4152526699058478,
  0.4736715102444958,
  0.48841850262278047,
  0.5048988560273049,
  0.4483490165206198,
  0.46002321731857304,
  0.45063035788488304,
  0.4917937725367031,
  0.43817286251781296,
